In [16]:
import os
import glob
import numpy as np
import pandas as pd
from pathlib import Path
from  vip_slap2_analysis.behavior import preprocess as ps

from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

C:\Users\andrew.shelton\AppData\Local\Temp\ipykernel_33908\421752032.py:8: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [17]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
basepath = r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics"

In [23]:
summary_path = glob.glob(os.path.join(basepath,'**summary.xlsx'))[0]
summary_df = pd.read_excel(summary_path,sheet_name='subjects')
session_df = pd.read_excel(summary_path,sheet_name='sessions')

In [24]:
target_mice = [826033]

In [25]:
process_df = session_df[(session_df['subject_id'].isin(target_mice))&(session_df['session_type']!='expression_check')]

In [26]:
process_df

,session_id,subject_id,session_#,session_date,indicator1,indicator2,dmd1_depth,dmd2_depth,paradigm,session_type,...,instrument_name,instrument_id,has raster ROI?,has integration roi?,behavior_rig,quality,flags,session_dir,purpose,notes
75,826033_2026-02-17_13-13-55,826033,2,2026-02-17,iGluSnFR4,RCaMP3,50,200,change_detection_passive,familiar,...,slap2_albert,SLAP2_1,yes,no,VCO.1,good,stimlus ID not recorded properly,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
76,826033_2026-02-18_11-57-04,826033,3,2026-02-18,iGluSnFR4,RCaMP3,50,200,change_detection_passive,familiar,...,slap2_albert,SLAP2_1,yes,no,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN


### Process DoC session collected w/ HARP/Bonsai

In [27]:
for idx,row in process_df.iterrows():
    try:
        harp_path = glob.glob(os.path.join(row['session_dir'],'**','**Behavior.harp'),recursive=True)[0]
        print(harp_path)
        if 'device.yml' in os.listdir(harp_path):
            print('metadata present')
        else:
            print('no metadata')
        if 'extracted_files' in os.listdir(harp_path):
            print('processed')
            pass
        else:
            print('processing...')
            try:
                ps.process_single_harp_session(harp_path)
            except:
                print('error processing')
        print('\n')
    except:
        
        print(f'No path for {row["session"]}','\n')

\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\826033\826033_2026-02-17_13-13-55\behavior\VCO1_Behavior.harp
metadata present
processed


\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\826033\826033_2026-02-18_11-57-04\behavior\VCO1_Behavior.harp
metadata present
processing...
Processing \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\826033\826033_2026-02-18_11-57-04\behavior\VCO1_Behavior.harp...
→ Saved data to \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\826033\826033_2026-02-18_11-57-04\behavior\VCO1_Behavior.harp\extracted_files




### QC behavior data

In [28]:
bad_sessions = []

for idx,row in process_df.iterrows():
    datapath = glob.glob(os.path.join(row['session_dir'],'**','**bonsai_event_log.csv'),recursive=True)[0]
    print(f"Checking data for {row['session_dir']}...")
    stim_df = pd.read_csv(datapath)
    num_stim_names = len([n for n in stim_df['Value'].unique() if '.tiff' in n])
    print(f'Number of unique stimulus IDs = {num_stim_names}')
    
    if num_stim_names<=1:
        print('STIMULUS ID NOT CORRECTLY RECORDED\n')
        bad_sessions.append(row['session_id'])
    else:
        print('QC passed')
#         print('QC passed, correcting image presentation times to HARP...')
#         pd_path = glob.glob(os.path.join(Path(datapath).parents[0],'extracted_files','photodiode.pkl'),recursive=True)[0]
#         photodiode_df = pd.read_pickle(pd_path)
        
#         stimulus_df = ps.correct_event_log(stim_df,photodiode_df,savepath = datapath)

Checking data for \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\826033\826033_2026-02-17_13-13-55...
Number of unique stimulus IDs = 1
STIMULUS ID NOT CORRECTLY RECORDED

Checking data for \\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\iGluSnFR4f+RCaMP3\826033\826033_2026-02-18_11-57-04...
Number of unique stimulus IDs = 7
QC passed
